# 第 2 课 A：看懂数组的形状与索引

**今天的核心问题：两条曲线，每条有 5 个频率值，怎样放进同一个数组？**

你已经会用列表生成和修改曲线。这一步用 NumPy 数组把数据排列清楚；下一步才转成 PyTorch 张量，随后学习归一化。

本次只要求能解释 `shape`、读出某个位置，并自己修改一个频率值。按顺序逐格运行，遇到“先预测”时先停一下。

**我搭好的部分：**Notebook 和可用环境、最小示例。  
**你要掌握的部分：**每条轴代表什么、索引取到什么、修改的是哪个数。  
本节使用 CPU 就够了；没有神经网络、梯度或训练。

## 0．接上上一课：显示出来，不等于返回出来

只执行 `print` 的函数可以在输出区显示列表，但如果没有执行 `return`，函数调用的值是 `None`。`None` 表示这里没有提供有效的数据结果，它不是空列表 `[]`，也不是数字 `0`。

下面不用重做作业，只观察两行输出属于谁。第一行来自函数内的打印，第二行检查调用结果。

In [1]:
def show_values():
    print([8, 9, 12, 9, 8])

result = show_values()
print("函数调用结果：", result)

[8, 9, 12, 9, 8]
函数调用结果： None


## 1．从一个列表开始

`import numpy as np`：加载 NumPy 数值计算库，用 `np` 作为短名字。  
`np.array(...)`：把给出的数组织成 NumPy 数组。  
`dtype=float`：指定用浮点数保存，允许小数；它是函数的命名参数。

这里的 5 个数，分别是在 0、0.1、0.2、0.3、0.4 秒时的频率，单位 kHz。

In [2]:
import numpy as np

frequency_list = [8, 9, 12, 9, 8]
frequency = np.array(frequency_list, dtype=float)
print(frequency)

[ 8.  9. 12.  9.  8.]


### 先预测：变成数组后，里面的数变了吗？

`8.` 是浮点数 8.0 的简洁显示，和原来的 8 数值相同。NumPy 打印数组时通常不显示元素之间的逗号；不要据此认为它变成了一个数。

数组可以把一个运算应用到每个元素上。下面将每个频率加 1 kHz，结果放进新变量 `shifted`。普通 Python 列表不能直接这样与数字相加。

In [3]:
shifted = frequency + 1
print("原来的频率：", frequency)
print("整体加 1：", shifted)

原来的频率： [ 8.  9. 12.  9.  8.]
整体加 1： [ 9. 10. 13. 10.  9.]


## 2．shape：不是数值，而是排列方式

`frequency.shape` 读取数组的**形状属性**，不需要再加括号。这里的点号表示“读取这个对象的某项属性”。  
`frequency.ndim` 是轴的数量；`frequency.size` 是所有元素的总数。

先运行，再读下面的解释。

In [4]:
print("shape：", frequency.shape)
print("ndim：", frequency.ndim)
print("size：", frequency.size)

shape： (5,)
ndim： 1
size： 5


这条曲线有 5 个数，但数组只有一条轴，所以形状是 `(5,)`，`ndim` 是 1。

括号中的形状用 **tuple（元组）** 表示，可以先把它当作一份有顺序的尺寸清单。`(5,)` 里的逗号表示只有一个元素的元组，后面没有“缺失的列数”。本次不要求记住元组的全部规则。

**`(5,)`、`(1, 5)` 和 `(5, 1)` 不相同。** 后两种都有两条轴；目前的 `frequency` 是一维数组，不能把它直接当成已有行、列两条轴的矩阵。

## 3．把两条曲线排成两行

每一行放一条完整曲线，每一列对应相同的一个时间点：

| 曲线 / 时间 | 0 s | 0.1 s | 0.2 s | 0.3 s | 0.4 s |
|---|---:|---:|---:|---:|---:|
| 第一条（行索引 0） | 8 | 9 | 12 | 9 | 8 |
| 第二条（行索引 1） | 8 | 9 | 10 | 9 | 8 |

外层 `[]` 收集两条曲线，里面的每个 `[]` 保存一条曲线。本例每行必须有相同数量的频率值，才能组成这样的普通二维数值数组。

`batch` 就是**一批样本**；我们约定“一条曲线是一个样本”。变量名 `batch` 是我们起的，不是 Python 特殊命令。

In [5]:
batch = np.array([
    [8, 9, 12, 9, 8],
    [8, 9, 10, 9, 8],
], dtype=float)

print(batch)
print("shape：", batch.shape)
print("ndim：", batch.ndim)
print("size：", batch.size)
print("len：", len(batch))

[[ 8.  9. 12.  9.  8.]
 [ 8.  9. 10.  9.  8.]]
shape： (2, 5)
ndim： 2
size： 10
len： 2


本例 `shape = (2, 5)`，按从左到右的顺序读：

- 第 0 条轴：2 个样本，也就是 2 行。
- 第 1 条轴：每条曲线 5 个时间点，也就是 5 列。

因此 `ndim = 2`，`size = 10`；`len(batch)` 只看第一条轴的长度，得到 2，不是 10。这里 batch size（批大小）是 2。

**轴的含义来自我们组织数据的约定。** `(2, 5)` 本身只说明尺寸，不会自动说明哪条轴是时间。它也没有记录秒、kHz 等单位，需要我们保留这些含义。

这些数是时间—频率轨迹上的频率值，不是声音波形的振幅，也不是完整的时频图。不要因为数据是二维就自动把它理解成一张声谱图。

## 4．定位一个数：先选曲线，再选时间点

沿用上一课的规则，索引从 **0** 开始。二维 NumPy 数组可以写成 `batch[行索引, 列索引]`。

先对照表格预测三行输出：

1. `batch[0]` 是第一条曲线，还是第一条曲线的第一个数？
2. `batch[1, 2]` 对应哪条曲线、哪个时间点？
3. `batch[:, 2]` 中的 `:` 表示这一轴全部取出，会得到哪些频率？

In [6]:
print("第一条曲线：", batch[0])
print("第二条曲线的第三个点：", batch[1, 2])
print("所有曲线的第三个点：", batch[:, 2])

第一条曲线： [ 8.  9. 12.  9.  8.]
第二条曲线的第三个点： 10.0
所有曲线的第三个点： [12. 10.]


只写 `batch[0]`，选中的是一整行；写 `batch[1, 2]`，选中一个数。`batch[:, 2]` 是“所有行、列索引 2”，对应两条曲线在 0.2 秒处的频率。

这里的一整行和一整列取出来都只有一条轴，形状都是 `(5,)` 或 `(2,)` 这种写法。下一步学习张量时，再比较保留一条长度为 1 的轴会有什么区别。

NumPy 的这类基本索引/切片可能与原数组共享数据。本节先只读取；想独立修改一份数据，使用下面的 `.copy()`。

## 5．你来改：只修改第二条曲线的峰值

**任务：**把第二条曲线在 0.2 秒时的频率由 10 改为 11 kHz，其他数不变。

先写预测，再动手：

- 应该使用哪两个索引？
- 改完以后，`shape` 和 `size` 会变吗？为什么？

下面 `.copy()` 是一个方法调用，括号表示执行复制，得到可独立修改的新数组。请在注释处补一行赋值，形式是 `edited[行索引, 列索引] = 新数值`；不要把中文占位词原样输入。

**提示：**这是第二条曲线、第三个点，但索引都从 0 开始。先试，不急着翻答案。

In [8]:
edited = batch.copy()

# 在这里补一行：只修改目标位置。
edited[1,2] = 11
print("修改后：")
print(edited)
print("shape：", edited.shape)
print("size：", edited.size)
print("原数组：")
print(batch)

修改后：
[[ 8.  9. 12.  9.  8.]
 [ 8.  9. 11.  9.  8.]]
shape： (2, 5)
size： 10
原数组：
[[ 8.  9. 12.  9.  8.]
 [ 8.  9. 10.  9.  8.]]


### 再做一个口头预测，暂时不用写代码

如果再加入一条有 5 个频率值的曲线，数组形状会是什么？增加的是哪条轴的长度？“多一条曲线”会让 `ndim` 增加吗？

请按自己的理解回答。我们用这个问题判断是否已经分清**轴的条数**和**每条轴的长度**。

## 本次停在这里

先把上面的小修改结果和解释发给我，不需要提前开始训练。当前只完成了第 2 课的数组部分；后面再用同一份数据学习 PyTorch 张量、数据类型和归一化。

你可以直接在下方 Markdown 格记录，或者填 `docs/学习记录.md`；不用重复填写两份。

### 我的练习记录（双击本格编辑）

- 我补的赋值语句：edited[1,2] = 11
- 修改后的目标值：11
- shape / size 是否变化及原因：不变，因为只动了对应位置的值，没有增删元素
- 新增第三条曲线后，shape 和 ndim 分别是什么：[3,5],3
- 我对 shape / 索引仍有的疑问：暂时没有

## 保存这次学习

运行修改过的格、确认输出与代码对应，然后保存 Notebook。提交仍由你练习；示例在 **PyCharm Terminal** 中运行，不在 Python 代码格中运行。

```powershell
git status
git add -- notebooks/02A_看懂数组形状与索引.ipynb README.md 计划.md docs/学习记录.md
git diff --cached --stat
git commit -m "Learn array shapes and indexing"
```

确认暂存范围正确后再 commit；需要同步时使用手册里的 push 流程。提交成功不代表新概念自动掌握。

可选查阅：[NumPy 数组入门](https://numpy.org/doc/stable/user/absolute_beginners.html)、[数组索引](https://numpy.org/doc/stable/user/basics.indexing.html)。课程使用已安装的 NumPy 2.2.5 验证；官方 stable 文档可能是较新版本。